# Electivo de Bioinformática — Clase 4

## Alineamiento de secuencias y procesamiento de archivos SAM/BAM

**Programa:** Doctorado — 2º año
**Duración:** 3 horas
**Fecha:** 7 de septiembre

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Retomar el ambiente `alineamiento` (con `bwa` y `samtools`) dejado listo en la clase 3 e incorporar una herramienta nueva: `fastqc`.
2. Obtener una referencia genómica (el genoma mitocondrial humano, rCRS/NC_012920.1) y datos FASTQ de ejemplo para practicar el flujo completo de alineamiento.
3. Ejecutar un control de calidad (QC) de datos de secuenciación crudos con **FastQC** e interpretar sus métricas principales.
4. Alinear lecturas FASTQ contra una referencia con `bwa index` y `bwa mem`, y reconocer los campos de un registro SAM (flag, CIGAR) en un caso real.
5. Convertir SAM → BAM, ordenar e indexar con `samtools`.
6. Explorar un BAM con `samtools flagstat`, `stats`, `view` (filtros por flag, extracción de regiones) y `mpileup`, dejando el terreno listo para el llamado de variantes de la próxima clase.

---
## 1. Repaso y preparación del ambiente (10 min)

En la clase 3 dejamos creado un ambiente de conda llamado **`alineamiento`**, con `bwa` y `samtools` ya instalados, y una referencia de ejemplo (`phix174.fasta`) lista para usar. Hoy vamos a:

- Reactivar ese ambiente y confirmar que `bwa`/`samtools` siguen funcionando.
- Agregar dos herramientas nuevas: **`fastqc`** (control de calidad de datos crudos) y **`dwgsim`** (simulador de lecturas, que usaremos solo para generar datos de ejemplo).

**Recordatorio de la clase 3:** cada celda `%%bash` de este notebook es una shell nueva e independiente. Por eso hay que hacer `source .../conda.sh` **y** `conda activate alineamiento` en cada celda que use conda o las herramientas instaladas en ese ambiente.

In [ ]:
%%bash
mkdir -p ~/bioinfo/clase4
cd ~/bioinfo/clase4

source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate alineamiento

echo "--- Herramientas instaladas en la clase 3 ---"
bwa 2>&1 | head -3
echo
samtools --version | head -1

echo
echo "--- Instalando herramientas nuevas para esta clase ---"
conda install -y fastqc dwgsim

echo
fastqc --version
dwgsim 2>&1 | head -3

---
## 2. Referencia mitocondrial humana y datos de ejemplo (30 min)

### 2.1 ¿Por qué el genoma mitocondrial humano?

Para practicar el flujo completo de alineamiento sin depender de descargas ni tiempos de cómputo largos, usaremos el **genoma mitocondrial humano de referencia (rCRS, *revised Cambridge Reference Sequence*, NC_012920.1)**:

- Tiene solo **16.569 pb** (miles de veces más chico que un cromosoma nuclear): todo el flujo corre en segundos.
- Es **haploide y circular**, lo que simplifica la interpretación de los resultados.
- Es un genoma real y muy usado en la práctica: genética forense, estudios de **poblaciones y haplogrupos mitocondriales**, y en clínica (enfermedades mitocondriales, donde además es clave el concepto de **heteroplasmia**: que una misma persona tenga una mezcla de moléculas de mtDNA normales y mutadas). Vamos a ver una asomo de esto último más adelante en la clase.

### 2.2 Descargar la referencia

In [ ]:
%%bash
cd ~/bioinfo/clase4
mkdir -p referencia && cd referencia

URL="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_012920.1&rettype=fasta&retmode=text"

echo "--- Intentando descargar la referencia mitocondrial humana (rCRS) desde NCBI ---"
if wget -q -O chrM.fasta "$URL" && [ -s chrM.fasta ] && grep -q "^>" chrM.fasta; then
    echo "Descarga exitosa."
else
    echo "AVISO: no se pudo descargar (revisen su conexion a internet)."
    echo "Usamos una copia local de la misma secuencia (rCRS, NC_012920.1) para poder seguir con la clase:"
    cat > chrM.fasta << 'EOF'
>NC_012920.1 Homo sapiens mitochondrion, complete genome (rCRS)
GATCACAGGTCTATCACCCTATTAACCACTCACGGGAGCTCTCCATGCATTTGGTATTTTCGTCTGGGGG
GTATGCACGCGATAGCATTGCGAGACGCTGGAGCCGGAGCACCCTATGTCGCAGTATCTGTCTTTGATTC
CTGCCTCATCCTATTATTTATCGCACCTACGTTCAATATTACAGGCGAACATACTTACTAAAGTGTGTTA
ATTAATTAATGCTTGTAGGACATAATAATAACAATTGAATGTCTGCACAGCCACTTTCCACACAGACATC
ATAACAAAAAATTTCCACCAAACCCCCCCTCCCCCGCTTCTGGCCACAGCACTTAAACACATCTCTGCCA
AACCCCAAAAACAAAGAACCCTAACACCAGCCTAACCAGATTTCAAATTTTATCTTTTGGCGGTATGCAC
TTTTAACAGTCACCCCCCAACTAACACATTATTTTCCCCTCCCACTCCCATACTACTAATCTCATCAATA
CAACCCCCGCCCATCCTACCCAGCACACACACACCGCTGCTAACCCCATACCCCGAACCAACCAAACCCC
AAAGACACCCCCCACAGTTTATGTAGCTTACCTCCTCAAAGCAATACACTGAAAATGTTTAGACGGGCTC
ACATCACCCCATAAACAAATAGGTTTGGTCCTAGCCTTTCTATTAGCTCTTAGTAAGATTACACATGCAA
GCATCCCCGTTCCAGTGAGTTCACCCTCTAAATCACCACGATCAAAAGGAACAAGCATCAAGCACGCAGC
AATGCAGCTCAAAACGCTTAGCCTAGCCACACCCCCACGGGAAACAGCAGTGATTAACCTTTAGCAATAA
ACGAAAGTTTAACTAAGCTATACTAACCCCAGGGTTGGTCAATTTCGTGCCAGCCACCGCGGTCACACGA
TTAACCCAAGTCAATAGAAGCCGGCGTAAAGAGTGTTTTAGATCACCCCCTCCCCAATAAAGCTAAAACT
CACCTGAGTTGTAAAAAACTCCAGTTGACACAAAATAGACTACGAAAGTGGCTTTAACATATCTGAACAC
ACAATAGCTAAGACCCAAACTGGGATTAGATACCCCACTATGCTTAGCCCTAAACCTCAACAGTTAAATC
AACAAAACTGCTCGCCAGAACACTACGAGCCACAGCTTAAAACTCAAAGGACCTGGCGGTGCTTCATATC
CCTCTAGAGGAGCCTGTTCTGTAATCGATAAACCCCGATCAACCTCACCACCTCTTGCTCAGCCTATATA
CCGCCATCTTCAGCAAACCCTGATGAAGGCTACAAAGTAAGCGCAAGTACCCACGTAAAGACGTTAGGTC
AAGGTGTAGCCCATGAGGTGGCAAGAAATGGGCTACATTTTCTACCCCAGAAAACTACGATAGCCCTTAT
GAAACTTAAGGGTCGAAGGTGGATTTAGCAGTAAACTAAGAGTAGAGTGCTTAGTTGAACAGGGCCCTGA
AGCGCGTACACACCGCCCGTCACCCTCCTCAAGTATACTTCAAAGGACATTTAACTAAAACCCCTACGCA
TTTATATAGAGGAGACAAGTCGTAACATGGTAAGTGTACTGGAAAGTGCACTTGGACGAACCAGAGTGTA
GCTTAACACAAAGCACCCAACTTACACTTAGGAGATTTCAACTTAACTTGACCGCTCTGAGCTAAACCTA
GCCCCAAACCCACTCCACCTTACTACCAGACAACCTTAGCCAAACCATTTACCCAAATAAAGTATAGGCG
ATAGAAATTGAAACCTGGCGCAATAGATATAGTACCGCAAGGGAAAGATGAAAAATTATAACCAAGCATA
ATATAGCAAGGACTAACCCCTATACCTTCTGCATAATGAATTAACTAGAAATAACTTTGCAAGGAGAGCC
AAAGCTAAGACCCCCGAAACCAGACGAGCTACCTAAGAACAGCTAAAAGAGCACACCCGTCTATGTAGCA
AAATAGTGGGAAGATTTATAGGTAGAGGCGACAAACCTACCGAGCCTGGTGATAGCTGGTTGTCCAAGAT
AGAATCTTAGTTCAACTTTAAATTTGCCCACAGAACCCTCTAAATCCCCTTGTAAATTTAACTGTTAGTC
CAAAGAGGAACAGCTCTTTGGACACTAGGAAAAAACCTTGTAGAGAGAGTAAAAAATTTAACACCCATAG
TAGGCCTAAAAGCAGCCACCAATTAAGAAAGCGTTCAAGCTCAACACCCACTACCTAAAAAATCCCAAAC
ATATAACTGAACTCCTCACACCCAATTGGACCAATCTATCACCCTATAGAAGAACTAATGTTAGTATAAG
TAACATGAAAACATTCTCCTCCGCATAAGCCTGCGTCAGATTAAAACACTGAACTGACAATTAACAGCCC
AATATCTACAATCAACCAACAAGTCATTATTACCCTCACTGTCAACCCAACACAGGCATGCTCATAAGGA
AAGGTTAAAAAAAGTAAAAGGAACTCGGCAAATCTTACCCCGCCTGTTTACCAAAAACATCACCTCTAGC
ATCACCAGTATTAGAGGCACCGCCTGCCCAGTGACACATGTTTAACGGCCGCGGTACCCTAACCGTGCAa
aggtagcataatcacttgttccttaaatagggacctgtatgaatggctccacgagggttcagctgtctct
tacttttaaccagtgaaattgacctgcccgtgaagaggcgggcataacacagcaagacgagaagacccta
tggagctttaatttaTTAATGCAAACAGTACCTAACAAACCCACAGGTCCTAAACTACCAAACCTGCATT
AAAAATTTCGGTTGGGGCGACCTCGGAGCAGAACCCAACCTCCGAGCAGTACATGCTAAGACTTCACCAG
TCAAAGCGAACTACTATACTCAATTGATCCAATAACTTGACCAACGGAACAAGTTACCCTAGGGATAACA
GCGCAATCCTATTCTAGAGTCCATATCAACAATAGGGTTTACGACCTCGATGTTGGATCAGGACATCCCG
ATGGTGCAGCCGCTATTAAAGGTTCGTTTGTTCAACGATTAAAGTCCTACGTGATCTGAGTTCAGACCGG
AGTAATCCAGGTCGGTTTCTATCTACNTTCAAATTCCTCCCTGTACGAAAGGACAAGAGAAATAAGGCCT
ACTTCACAAAGCGCCTTCCCCCGTAAATGATATCATCTCAACTTAGTATTATACCCACACCCACCCAAGA
ACAGGGTTTgttaagatggcagagcccggtaatcgcataaaacttaaaactttacagtcagaggttcaat
tcctcttcttaacaacaTACCCATGGCCAACCTCCTACTCCTCATTGTACCCATTCTAATCGCAATGGCA
TTCCTAATGCTTACCGAACGAAAAATTCTAGGCTATATACAACTACGCAAAGGCCCCAACGTTGTAGGCC
CCTACGGGCTACTACAACCCTTCGCTGACGCCATAAAACTCTTCACCAAAGAGCCCCTAAAACCCGCCAC
ATCTACCATCACCCTCTACATCACCGCCCCGACCTTAGCTCTCACCATCGCTCTTCTACTATGAACCCCC
CTCCCCATACCCAACCCCCTGGTCAACCTCAACCTAGGCCTCCTATTTATTCTAGCCACCTCTAGCCTAG
CCGTTTACTCAATCCTCTGATCAGGGTGAGCATCAAACTCAAACTACGCCCTGATCGGCGCACTGCGAGC
AGTAGCCCAAACAATCTCATATGAAGTCACCCTAGCCATCATTCTACTATCAACATTACTAATAAGTGGC
TCCTTTAACCTCTCCACCCTTATCACAACACAAGAACACCTCTGATTACTCCTGCCATCATGACCCTTGG
CCATAATATGATTTATCTCCACACTAGCAGAGACCAACCGAACCCCCTTCGACCTTGCCGAAGGGGAGTC
CGAACTAGTCTCAGGCTTCAACATCGAATACGCCGCAGGCCCCTTCGCCCTATTCTTCATAGCCGAATAC
ACAAACATTATTATAATAAACACCCTCACCACTACAATCTTCCTAGGAACAACATATGACGCACTCTCCC
CTGAACTCTACACAACATATTTTGTCACCAAGACCCTACTTCTAACCTCCCTGTTCTTATGAATTCGAAC
AGCATACCCCCGATTCCGCTACGACCAACTCATACACCTCCTATGAAAAAACTTCCTACCACTCACCCTA
GCATTACTTATATGATATGTCTCCATACCCATTACAATCTCCAGCATTCCCCCTCAAACCTAAGAAATAT
GTCTGATAAAAGAGTTACTTTGATAGAGTAAATAATAGGAGCTTAAACCCCCTTATTTctaggactatga
gaatcgaacccatccctgagaatccaaaattctccgtgccacctatcacaccccatcctaAAGTAAGGTC
AGCTAAATAAGCTATCGGGCCCATACCCCGAAAATGTTGGTTATACCCTTCCCGTACTAATTAATCCCCT
GGCCCAACCCGTCATCTACTCTACCATCTTTGCAGGCACACTCATCACAGCGCTAAGCTCGCACTGATTT
TTTACCTGAGTAGGCCTAGAAATAAACATGCTAGCTTTTATTCCAGTTCTAACCAAAAAAATAAACCCTC
GTTCCACAGAAGCTGCCATCAAGTATTTCCTCACGCAAGCAACCGCATCCATAATCCTTCTAATAGCTAT
CCTCTTCAACAATATACTCTCCGGACAATGAACCATAACCAATACTACCAATCAATACTCATCATTAATA
ATCATAATAGCTATAGCAATAAAACTAGGAATAGCCCCCTTTCACTTCTGAGTCCCAGAGGTTACCCAAG
GCACCCCTCTGACATCCGGCCTGCTTCTTCTCACATGACAAAAACTAGCCCCCATCTCAATCATATACCA
AATCTCTCCCTCACTAAACGTAAGCCTTCTCCTCACTCTCTCAATCTTATCCATCATAGCAGGCAGTTGA
GGTGGATTAAACCAAACCCAGCTACGCAAAATCTTAGCATACTCCTCAATTACCCACATAGGATGAATAA
TAGCAGTTCTACCGTACAACCCTAACATAACCATTCTTAATTTAACTATTTATATTATCCTAACTACTAC
CGCATTCCTACTACTCAACTTAAACTCCAGCACCACGACCCTACTACTATCTCGCACCTGAAACAAGCTA
ACATGACTAACACCCTTAATTCCATCCACCCTCCTCTCCCTAGGAGGCCTGCCCCCGCTAACCGGCTTTT
TGCCCAAATGGGCCATTATCGAAGAATTCACAAAAAACAATAGCCTCATCATCCCCACCATCATAGCCAC
CATCACCCTCCTTAACCTCTACTTCTACCTACGCCTAATCTACTCCACCTCAATCACACTACTCCCCATA
TCTAACAACGTAAAAATAAAATGACAGTTTGAACATACAAAACCCACCCCATTCCTCCCCACACTCATCG
CCCTTACCACGCTACTCCTACCTATCTCCCCTTTTATACTAATAATCTTATAGAAATTTAGGTTAAATAC
AGACCAAGAGCCTTCAAAGCCCTCAGTAAGTTGCAATACTTAATTTCTGTAACAGCTAAGGACTGCAAAA
CCCCACTCTGCATCAACTGAACGCAAATCAGCCACTTTAATTAAGCTAAGCCCTTACTAGACCAATGGGA
CTTAAACCCACAAACACTTAGTTAACAGCTAAGCACCCTAATCAACTGGCTTCAATCTACTTCTCCCGCC
GCCGGGAAAAAAGGCGGGAGAAGCCCCGGCAGGTTTGAAGCTGCTTCTTCGAATTTGCAATTCAATATGA
AAATCACCTCGGAGCTGGTAAAAAGAGGCCTAACCCCTGTCTTTAGATTTACAGTCCAATGCTTCACTCA
GCCATTTTACCTCACCCCCACTGATGTTCGCCGACCGTTGACTATTCTCTACAAACCACAAAGACATTGG
AACACTATACCTATTATTCGGCGCATGAGCTGGAGTCCTAGGCACAGCTCTAAGCCTCCTTATTCGAGCC
GAGCTGGGCCAGCCAGGCAACCTTCTAGGTAACGACCACATCTACAACGTTATCGTCACAGCCCATGCAT
TTGTAATAATCTTCTTCATAGTAATACCCATCATAATCGGAGGCTTTGGCAACTGACTAGTTCCCCTAAT
AATCGGTGCCCCCGATATGGCGTTTCCCCGCATAAACAACATAAGCTTCTGACTCTTACCTCCCTCTCTC
CTACTCCTGCTCGCATCTGCTATAGTGGAGGCCGGAGCAGGAACAGGTTGAACAGTCTACCCTCCCTTAG
CAGGGAACTACTCCCACCCTGGAGCCTCCGTAGACCTAACCATCTTCTCCTTACACCTAGCAGGTGTCTC
CTCTATCTTAGGGGCCATCAATTTCATCACAACAATTATCAATATAAAACCCCCTGCCATAACCCAATAC
CAAACGCCCCTCTTCGTCTGATCCGTCCTAATCACAGCAGTCCTACTTCTCCTATCTCTCCCAGTCCTAG
CTGCTGGCATCACTATACTACTAACAGACCGCAACCTCAACACCACCTTCTTCGACCCCGCCGGAGGAGG
AGACCCCATTCTATACCAACACCTATTCTGATTTTTCGGTCACCCTGAAGTTTATATTCTTATCCTACCA
GGCTTCGGAATAATCTCCCATATTGTAACTTACTACTCCGGAAAAAAAGAACCATTTGGATACATAGGTA
TGGTCTGAGCTATGATATCAATTGGCTTCCTAGGGTTTATCGTGTGAGCACACCATATATTTACAGTAGG
AATAGACGTAGACACACGAGCATATTTCACCTCCGCTACCATAATCATCGCTATCCCCACCGGCGTCAAA
GTATTTAGCTGACTCGCCACACTCCACGGAAGCAATATGAAATGATCTGCTGCAGTGCTCTGAGCCCTAG
GATTCATCTTTCTTTTCACCGTAGGTGGCCTGACTGGCATTGTATTAGCAAACTCATCACTAGACATCGT
ACTACACGACACGTACTACGTTGTAGCCCACTTCCACTATGTCCTATCAATAGGAGCTGTATTTGCCATC
ATAGGAGGCTTCATTCACTGATTTCCCCTATTCTCAGGCTACACCCTAGACCAAACCTACGCCAAAATCC
ATTTCACTATCATATTCATCGGCGTAAATCTAACTTTCTTCCCACAACACTTTCTCGGCCTATCCGGAAT
GCCCCGACGTTACTCGGACTACCCCGATGCATACACCACATGAAACATCCTATCATCTGTAGGCTCATTC
ATTTCTCTAACAGCAGTAATATTAATAATTTTCATGATTTGAGAAGCCTTCGCTTCGAAGCGAAAAGTCC
TAATAGTAGAAGAACCCTCCATAAACCTGGAGTGACTATATGGATGCCCCCCACCCTACCACACATTCGA
AGAACCCGTATACATAAAATCTAGACAaaaaaggaaggaatcgaaccccccaaagctggtttcaagccaa
ccccatggcctccatgactttttcAAAAAGGTATTAGAAAAACCATTTCATAACTTTGTCAAAGTTAAAT
TATAGGCTAAATCCTATATATCTTAATGGCACATGCAGCGCAAGTAGGTCTACAAGACGCTACTTCCCCT
ATCATAGAAGAGCTTATCACCTTTCATGATCACGCCCTCATAATCATTTTCCTTATCTGCTTCCTAGTCC
TGTATGCCCTTTTCCTAACACTCACAACAAAACTAACTAATACTAACATCTCAGACGCTCAGGAAATAGA
AACCGTCTGAACTATCCTGCCCGCCATCATCCTAGTCCTCATCGCCCTCCCATCCCTACGCATCCTTTAC
ATAACAGACGAGGTCAACGATCCCTCCCTTACCATCAAATCAATTGGCCACCAATGGTACTGAACCTACG
AGTACACCGACTACGGCGGACTAATCTTCAACTCCTACATACTTCCCCCATTATTCCTAGAACCAGGCGA
CCTGCGACTCCTTGACGTTGACAATCGAGTAGTACTCCCGATTGAAGCCCCCATTCGTATAATAATTACA
TCACAAGACGTCTTGCACTCATGAGCTGTCCCCACATTAGGCTTAAAAACAGATGCAATTCCCGGACGTC
TAAACCAAACCACTTTCACCGCTACACGACCGGGGGTATACTACGGTCAATGCTCTGAAATCTGTGGAGC
AAACCACAGTTTCATGCCCATCGTCCTAGAATTAATTCCCCTAAAAATCTTTGAAATAGGGCCCGTATTT
ACCCTATAGCACCCCCTCTACCCCCTCTAGAGCCCACTGTAAAGCTAACTTAGCATTAACCTTTTAAGTT
AAAGATTAAGAGAACCAACACCTCTTTACAGTGAAATGCCCCAACTAAATACTACCGTATGGCCCACCAT
AATTACCCCCATACTCCTTACACTATTCCTCATCACCCAACTAAAAATATTAAACACAAACTACCACCTA
CCTCCCTCACCAAAGCCCATAAAAATAAAAAATTATAACAAACCCTGAGAACCAAAATGAACGAAAATCT
GTTCGCTTCATTCATTGCCCCCACAATCCTAGGCCTACCCGCCGCAGTACTGATCATTCTATTTCCCCCT
CTATTGATCCCCACCTCCAAATATCTCATCAACAACCGACTAATCACCACCCAACAATGACTAATCAAAC
TAACCTCAAAACAAATGATAACCATACACAACACTAAAGGACGAACCTGATCTCTTATACTAGTATCCTT
AATCATTTTTATTGCCACAACTAACCTCCTCGGACTCCTGCCTCACTCATTTACACCAACCACCCAACTA
TCTATAAACCTAGCCATGGCCATCCCCTTATGAGCGGGCACAGTGATTATAGGCTTTCGCTCTAAGATTA
AAAATGCCCTAGCCCACTTCTTACCACAAGGCACACCTACACCCCTTATCCCCATACTAGTTATTATCGA
AACCATCAGCCTACTCATTCAACCAATAGCCCTGGCCGTACGCCTAACCGCTAACATTACTGCAGGCCAC
CTACTCATGCACCTAATTGGAAGCGCCACCCTAGCAATATCAACCATTAACCTTCCCTCTACACTTATCA
TCTTCACAATTCTAATTCTACTGACTATCCTAGAAATCGCTGTCGCCTTAATCCAAGCCTACGTTTTCAC
ACTTCTAGTAAGCCTCTACCTGCACGACAACACATAATGACCCACCAATCACATGCCTATCATATAGTAA
AACCCAGCCCATGACCCCTAACAGGGGCCCTCTCAGCCCTCCTAATGACCTCCGGCCTAGCCATGTGATT
TCACTTCCACTCCATAACGCTCCTCATACTAGGCCTACTAACCAACACACTAACCATATACCAATGATGG
CGCGATGTAACACGAGAAAGCACATACCAAGGCCACCACACACCACCTGTCCAAAAAGGCCTTCGATACG
GGATAATCCTATTTATTACCTCAGAAGTTTTTTTCTTCGCAGGATTTTTCTGAGCCTTTTACCACTCCAG
CCTAGCCCCTACCCCCCAATTAGGAGGGCACTGGCCCCCAACAGGCATCACCCCGCTAAATCCCCTAGAA
GTCCCACTCCTAAACACATCCGTATTACTCGCATCAGGAGTATCAATCACCTGAGCTCACCATAGTCTAA
TAGAAAACAACCGAAACCAAATAATTCAAGCACTGCTTATTACAATTTTACTGGGTCTCTATTTTACCCT
CCTACAAGCCTCAGAGTACTTCGAGTCTCCCTTCACCATTTCCGACGGCATCTACGGCTCAACATTTTTT
GTAGCCACAGGCTTCCACGGACTTCACGTCATTATTGGCTCAACTTTCCTCACTATCTGCTTCATCCGCC
AACTAATATTTCACTTTACATCCAAACATCACTTTGGCTTCGAAGCCGCCGCCTGATACTGGCATTTTGT
AGATGTGGTTTGACTATTTCTGTATGTCTCCATCTATTGATGAGGGTCTTACTCTTTTAGTATAAATAGT
ACCGTTAACTTCCAATTAACTAGTTTTGACAACATTCAAAAAAGAGTAATAAACTTCGCCTTAATTTTAA
TAATCAACACCCTCCTAGCCTTACTACTAATAATTATTACATTTTGACTACCACAACTCAACGGCTACAT
AGAAAAATCCACCCCTTACGAGTGCGGCTTCGACCCTATATCCCCCGCCCGCGTCCCTTTCTCCATAAAA
TTCTTCTTAGTAGCTATTACCTTCTTATTATTTGATCTAGAAATTGCCCTCCTTTTACCCCTACCATGAG
CCCTACAAACAACTAACCTGCCACTAATAGTTATGTCATCCCTCTTATTAATCATCATCCTAGCCCTAAG
TCTGGCCTATGAGTGACTACAAAAAGGATTAGACTGAACCGAATTGGTATATAGTTTAAACAAAACGAAT
GATTTCGACTCATTAAATTATGATAATCATATTTACCAAATGCCCCTCATTTACATAAATATTATACTAG
CATTTACCATCTCACTTCTAGGAATACTAGTATATCGCTCACACCTCATATCCTCCCTACTATGCCTAGA
AGGAATAATACTATCGCTGTTCATTATAGCTACTCTCATAACCCTCAACACCCACTCCCTCTTAGCCAAT
ATTGTGCCTATTGCCATACTAGTCTTTGCCGCCTGCGAAGCAGCGGTGGGCCTAGCCCTACTAGTCTCAA
TCTCCAACACATATGGCCTAGACTACGTACATAACCTAAACCTACTCCAATGCTAAAACTAATCGTCCCA
ACAATTATATTACTACCACTGACATGACTTTCCAAAAAACACATAATTTGAATCAACACAACCACCCACA
GCCTAATTATTAGCATCATCCCTCTACTATTTTTTAACCAAATCAACAACAACCTATTTAGCTGTTCCCC
AACCTTTTCCTCCGACCCCCTAACAACCCCCCTCCTAATACTAACTACCTGACTCCTACCCCTCACAATC
ATGGCAAGCCAACGCCACTTATCCAGTGAACCACTATCACGAAAAAAACTCTACCTCTCTATACTAATCT
CCCTACAAATCTCCTTAATTATAACATTCACAGCCACAGAACTAATCATATTTTATATCTTCTTCGAAAC
CACACTTATCCCCACCTTGGCTATCATCACCCGATGAGGCAACCAGCCAGAACGCCTGAACGCAGGCACA
TACTTCCTATTCTACACCCTAGTAGGCTCCCTTCCCCTACTCATCGCACTAATTTACACTCACAACACCC
TAGGCTCACTAAACATTCTACTACTCACTCTCACTGCCCAAGAACTATCAAACTCCTGAGCCAACAACTT
AATATGACTAGCTTACACAATAGCTTTTATAGTAAAGATACCTCTTTACGGACTCCACTTATGACTCCCT
AAAGCCCATGTCGAAGCCCCCATCGCTGGGTCAATAGTACTTGCCGCAGTACTCTTAAAACTAGGCGGCT
ATGGTATAATACGCCTCACACTCATTCTCAACCCCCTGACAAAACACATAGCCTACCCCTTCCTTGTACT
ATCCCTATGAGGCATAATTATAACAAGCTCCATCTGCCTACGACAAACAGACCTAAAATCGCTCATTGCA
TACTCTTCAATCAGCCACATAGCCCTCGTAGTAACAGCCATTCTCATCCAAACCCCCTGAAGCTTCACCG
GCGCAGTCATTCTCATAATCGCCCACGGGCTTACATCCTCATTACTATTCTGCCTAGCAAACTCAAACTA
CGAACGCACTCACAGTCGCATCATAATCCTCTCTCAAGGACTTCAAACTCTACTCCCACTAATAGCTTTT
TGATGACTTCTAGCAAGCCTCGCTAACCTCGCCTTACCCCCCACTATTAACCTACTGGGAGAACTCTCTG
TGCTAGTAACCACGTTCTCCTGATCAAATATCACTCTCCTACTTACAGGACTCAACATACTAGTCACAGC
CCTATACTCCCTCTACATATTTACCACAACACAATGGGGCTCACTCACCCACCACATTAACAACATAAAA
CCCTCATTCACACGAGAAAACACCCTCATGTTCATACACCTATCCCCCATTCTCCTCCTATCCCTCAACC
CCGACATCATTACCGGGTTTTCCTCTTGTAAATATAGTTTAACCAAAACATCAGATTGTGAATCTGACAA
CAGAGGCTTACGACCCCTTATTTACCGAGAAAGCTCACAAGAACTGCTAACTCATGCCCCCATGTCTAAC
AACATGGCTTTCTCAACTTTTAAAGGATAACAGCTATCCATTGGTCTTAGGCCCCAAAAATTTTGGTGCA
ACTCCAAATAAAAGTAATAACCATGCACACTACTATAACCACCCTAACCCTGACTTCCCTAATTCCCCCC
ATCCTTACCACCCTCGTTAACCCTAACAAAAAAAACTCATACCCCCATTATGTAAAATCCATTGTCGCAT
CCACCTTTATTATCAGTCTCTTCCCCACAACAATATTCATGTGCCTAGACCAAGAAGTTATTATCTCGAA
CTGACACTGAGCCACAACCCAAACAACCCAGCTCTCCCTAAGCTTCAAACTAGACTACTTCTCCATAATA
TTCATCCCTGTAGCATTGTTCGTTACATGGTCCATCATAGAATTCTCACTGTGATATATAAACTCAGACC
CAAACATTAATCAGTTCTTCAAATATCTACTCATCTTCCTAATTACCATACTAATCTTAGTTACCGCTAA
CAACCTATTCCAACTGTTCATCGGCTGAGAGGGCGTAGGAATTATATCCTTCTTGCTCATCAGTTGATGA
TACGCCCGAGCAGATGCCAACACAGCAGCCATTCAAGCAATCCTATACAACCGTATCGGCGATATCGGTT
TCATCCTCGCCTTAGCATGATTTATCCTACACTCCAACTCATGAGACCCACAACAAATAGCCCTTCTAAA
CGCTAATCCAAGCCTCACCCCACTACTAGGCCTCCTCCTAGCAGCAGCAGGCAAATCAGCCCAATTAGGT
CTCCACCCCTGACTCCCCTCAGCCATAGAAGGCCCCACCCCAGTCTCAGCCCTACTCCACTCAAGCACTA
TAGTTGTAGCAGGAATCTTCTTACTCATCCGCTTCCACCCCCTAGCAGAAAATAGCCCACTAATCCAAAC
TCTAACACTATGCTTAGGCGCTATCACCACTCTGTTCGCAGCAGTCTGCGCCCTTACACAAAATGACATC
AAAAAAATCGTAGCCTTCTCCACTTCAAGTCAACTAGGACTCATAATAGTTACAATCGGCATCAACCAAC
CACACCTAGCATTCCTGCACATCTGTACCCACGCCTTCTTCAAAGCCATACTATTTATGTGCTCCGGGTC
CATCATCCACAACCTTAACAATGAACAAGATATTCGAAAAATAGGAGGACTACTCAAAACCATACCTCTC
ACTTCAACCTCCCTCACCATTGGCAGCCTAGCATTAGCAGGAATACCTTTCCTCACAGGTTTCTACTCCA
AAGACCACATCATCGAAACCGCAAACATATCATACACAAACGCCTGAGCCCTATCTATTACTCTCATCGC
TACCTCCCTGACAAGCGCCTATAGCACTCGAATAATTCTTCTCACCCTAACAGGTCAACCTCGCTTCCCC
ACCCTTACTAACATTAACGAAAATAACCCCACCCTACTAAACCCCATTAAACGCCTGGCAGCCGGAAGCC
TATTCGCAGGATTTCTCATTACTAACAACATTTCCCCCGCATCCCCCTTCCAAACAACAATCCCCCTCTA
CCTAAAACTCACAGCCCTCGCTGTCACTTTCCTAGGACTTCTAACAGCCCTAGACCTCAACTACCTAACC
AACAAACTTAAAATAAAATCCCCACTATGCACATTTTATTTCTCCAACATACTCGGATTCTACCCTAGCA
TCACACACCGCACAATCCCCTATCTAGGCCTTCTTACGAGCCAAAACCTGCCCCTACTCCTCCTAGACCT
AACCTGACTAGAAAAGCTATTACCTAAAACAATTTCACAGCACCAAATCTCCACCTCCATCATCACCTCA
ACCCAAAAAGGCATAATTAAACTTTACTTCCTCTCTTTCTTCTTCCCACTCATCCTAACCCTACTCCTAA
TCACATAACCTATTCCCCCGAGCAATCTCAATTACAATATATACACCAACAAACAATGTTCAACCAGTAA
CTACTACTAATCAACGCCCATAATCATACAAAGCCCCCGCACCAATAGGATCCTCCCGAATCAACCCTGA
CCCCTCTCCTTCATAAATTATTCAGCTTCCTACACTATTAAAGTTTACCACAACCACCACCCCATCATAC
TCTTTCACCCACAGCACCAATCCTACCTCCATCGCTAACCCCACTAAAACACTCACCAAGACCTCAACCC
CTGACCCCCATGCCTCAGGATACTCCTCAATAGCCATCGCTGTAGTATATCCAAAGACAACCATCATTCC
CCCTAAATAAATTAAAAAAACTATTAAACCCATATAACCTCCCCCAAAATTCAGAATAATAACACACCCG
ACCACACCGCTAACAATCAATACTAAACCCCCATAAATAGGAGAAGGCTTAGAAGAAAACCCCACAAACC
CCATTACTAAACCCACACTCAACAGAAACAAAGCATACATCATTATTCTCGCACGGACTACAACCACGAC
CAATGATATGAAAAACCATCGTTGTATTTCAACTACAAGAACACCAATGACCCCAATACGCAAAACTAAC
CCCCTAATAAAATTAATTAACCACTCATTCATCGACCTCCCCACCCCATCCAACATCTCCGCATGATGAA
ACTTCGGCTCACTCCTTGGCGCCTGCCTGATCCTCCAAATCACCACAGGACTATTCCTAGCCATGCACTA
CTCACCAGACGCCTCAACCGCCTTTTCATCAATCGCCCACATCACTCGAGACGTAAATTATGGCTGAATC
ATCCGCTACCTTCACGCCAATGGCGCCTCAATATTCTTTATCTGCCTCTTCCTACACATCGGGCGAGGCC
TATATTACGGATCATTTCTCTACTCAGAAACCTGAAACATCGGCATTATCCTCCTGCTTGCAACTATAGC
AACAGCCTTCATAGGCTATGTCCTCCCGTGAGGCCAAATATCATTCTGAGGGGCCACAGTAATTACAAAC
TTACTATCCGCCATCCCATACATTGGGACAGACCTAGTTCAATGAATCTGAGGAGGCTACTCAGTAGACA
GTCCCACCCTCACACGATTCTTTACCTTTCACTTCATCTTGCCCTTCATTATTGCAGCCCTAGCAACACT
CCACCTCCTATTCTTGCACGAAACGGGATCAAACAACCCCCTAGGAATCACCTCCCATTCCGATAAAATC
ACCTTCCACCCTTACTACACAATCAAAGACGCCCTCGGCTTACTTCTCTTCCTTCTCTCCTTAATGACAT
TAACACTATTCTCACCAGACCTCCTAGGCGACCCAGACAATTATACCCTAGCCAACCCCTTAAACACCCC
TCCCCACATCAAGCCCGAATGATATTTCCTATTCGCCTACACAATTCTCCGATCCGTCCCTAACAAACTA
GGAGGCGTCCTTGCCCTATTACTATCCATCCTCATCCTAGCAATAATCCCCATCCTCCATATATCCAAAC
AACAAAGCATAATATTTCGCCCACTAAGCCAATCACTTTATTGACTCCTAGCCGCAGACCTCCTCATTCT
AACCTGAATCGGAGGACAACCAGTAAGCTACCCTTTTACCATCATTGGACAAGTAGCATCCGTACTATAC
TTCACAACAATCCTAATCCTAATACCAACTATCTCCCTAATTGAAAACAAAATACTCAAATGGGCCTGTC
CTTGTAGTATAAACTAATACACCAGTCTTGTAAACCGGAGATGAAAACCTTTTTCCAAGGACAAATCAGA
GAAAAAGTCTTTAACTCCACCATTAGCACCCAAAGCTAAGATTCTAATTTAAACTATTCTCTGTTCTTTC
ATGGGGAAGCAGATTTGGGTACCACCCAAGTATTGACTCACCCATCAACAACCGCTATGTATTTCGTACA
TTACTGCCAGCCACCATGAATATTGTACGGTACCATAAATACTTGACCACCTGTAGTACATAAAAACCCA
ATCCACATCAAAACCCCCTCCCCATGCTTACAAGcaagtacagcaatcaaccctcaactatcacacatca
actgcaactCCAAAGCCACCCCTCACCCACTAGGATACCAACAAACCTACCCACCCTTAACAGTACATAG
TACATAAAGCCATTTACCGTACATAGCACATTACAGTCAAATCCCTTCTCGTCCCCATGGATGACCCCCC
TCAGATAGGGGTCCCTTGACCACCATCCTCCGTGAAATCAATATCCCGCACAAGAGTGCTACTCTCCTCG
CTCCGGGCCCATAACACTTGGGGGTAGCTAAAGTGAACTGTATCCGACATCTGGTTCCTACTTCAGGGTC
ATAAAGCCTAAATAGCCCACACGTTCCCCTTAAATAAGACATCACGATG
EOF
fi

echo
echo "--- Verificamos la referencia ---"
echo "Numero de secuencias:"; grep -c "^>" chrM.fasta
echo "Largo total (pb):"; grep -v "^>" chrM.fasta | tr -d '\n' | wc -c
head -1 chrM.fasta

### 2.3 Indexar la referencia

Antes de alinear, `bwa` necesita construir su propio índice (estructura tipo Burrows-Wheeler) y `samtools` necesita el índice `.fai` que ya vimos en la clase 3 (acceso rápido a regiones).

In [ ]:
%%bash
cd ~/bioinfo/clase4/referencia

echo "--- Indice .fai (samtools) ---"
samtools faidx chrM.fasta
cat chrM.fasta.fai

echo
echo "--- Indice de bwa (Burrows-Wheeler) ---"
bwa index chrM.fasta
ls -la chrM.fasta.*

> La columna del `.fai` son: nombre de la secuencia, largo total (pb), posición en bytes donde empieza la secuencia en el archivo, bases por línea y bytes por línea — con esto `samtools` puede saltar directo a cualquier región sin leer el archivo completo.

### 2.4 Datos de secuenciación de ejemplo: simulación de lecturas

Todavía no tenemos un FASTQ para practicar. En vez de depender de descargar datos reales (pesados y con consideraciones de privacidad/consentimiento), vamos a **simular** lecturas cortas a partir de la referencia con `dwgsim`.

La simulación de lecturas es una técnica real y muy usada en bioinformática: sirve para **probar y validar pipelines** (si sé exactamente qué le puse a la referencia, puedo comprobar si mi flujo lo recupera correctamente), para **comparar herramientas** (benchmarking) y, como hoy, para **enseñar** con datos reproducibles.

`dwgsim` simula lecturas pareadas (paired-end) a partir de un FASTA, introduciendo errores de secuenciación al azar y, opcionalmente, mutaciones puntuales respecto a la referencia — y guarda un archivo con la posición exacta de cada mutación que introdujo, que funciona como **"respuesta correcta"**.

In [ ]:
%%bash
cd ~/bioinfo/clase4
mkdir -p fastq && cd fastq

dwgsim -N 2000 -1 100 -2 100 -e 0.001 -E 0.001 -r 0.0006 -R 0 -X 0 -y 0 -z 7 -c 0 \
    ../referencia/chrM.fasta mtDNA_sample

# Renombramos a la convencion estandar R1/R2 y limpiamos un formato que no usaremos hoy (bfast)
mv mtDNA_sample.bwa.read1.fastq.gz mtDNA_sample_R1.fastq.gz
mv mtDNA_sample.bwa.read2.fastq.gz mtDNA_sample_R2.fastq.gz
rm -f mtDNA_sample.bfast.fastq.gz

echo "--- Archivos generados ---"
ls -la

echo
echo "--- Numero de pares de lecturas ---"
echo "$(zcat mtDNA_sample_R1.fastq.gz | wc -l) / 4" | bc

Parámetros usados: `-N 2000` (2000 pares de lecturas), `-1 100 -2 100` (100 pb cada lectura, R1 y R2), `-e`/`-E` (tasa de error de secuenciación por base en R1/R2, 0.1%), `-r` (tasa de mutación puntual respecto a la referencia), `-R 0` (0% de esas mutaciones son indeles: solo generamos SNPs para simplificar), `-z 7` (semilla aleatoria, para que el resultado sea reproducible), `-c 0` (modelo de error estilo Illumina).

`dwgsim` dejó además dos archivos con las mutaciones que introdujo: `mtDNA_sample.mutations.txt` y `mtDNA_sample.mutations.vcf`. **Guárdenlos, pero no los abran todavía** — son la respuesta correcta que vamos a comparar con lo que encuentre un llamador de variantes real en la próxima clase.

---
## 3. Control de calidad con FastQC (30 min)

Antes de alinear cualquier dato real, lo primero que se hace **siempre** es revisar la calidad de las lecturas crudas. **FastQC** es la herramienta estándar para esto: lee un FASTQ y genera un reporte HTML con ~10 módulos, cada uno con un semáforo (✔ verde / ⚠ amarillo / ✘ rojo):

| Módulo | Qué evalúa |
|---|---|
| Per base sequence quality | Calidad (Phred) por posición a lo largo de la lectura |
| Per sequence quality scores | Distribución de la calidad promedio por lectura |
| Per base sequence content | % de cada base (A/C/G/T) por posición |
| Per sequence GC content | Distribución de %GC comparada con una normal teórica |
| Sequence Duplication Levels | Cuántas lecturas están duplicadas |
| Overrepresented sequences | Secuencias que aparecen con frecuencia anormalmente alta (posibles adaptadores o contaminación) |
| Adapter Content | Presencia de adaptadores de secuenciación sin recortar |

Un ⚠/✘ **no significa automáticamente que los datos estén mal** — depende del tipo de muestra y de la pregunta biológica. Parte del trabajo es saber cuándo una alerta es esperable y cuándo es señal de un problema real.

In [ ]:
%%bash
cd ~/bioinfo/clase4
mkdir -p fastqc_reportes

fastqc -o fastqc_reportes fastq/mtDNA_sample_R1.fastq.gz fastq/mtDNA_sample_R2.fastq.gz

echo
ls fastqc_reportes

**Para ver el reporte en Colab:** en el panel de archivos (ícono de carpeta a la izquierda), naveguen a `bioinfo/clase4/fastqc_reportes/`, click derecho sobre el `.html` → *Download*, y ábranlo en el navegador. También pueden descargarlo por código con `from google.colab import files; files.download("ruta/al/archivo.html")`.

**Qué deberían ver con estos datos simulados** (2000 lecturas de 100 pb, %GC≈44%, coherente con el genoma mitocondrial humano real):

- ✔ **Per base sequence quality** y **Per sequence quality scores**: calidad alta y pareja en toda la lectura — simulamos una tasa de error baja y constante (0.1%), sin la caída de calidad hacia el extremo 3' que sí se ve en datos reales de un secuenciador Illumina.
- ✔ **Adapter Content**: sin adaptadores, porque no simulamos fragmentos más cortos que la lectura.
- ⚠ **Per sequence GC content** y **Overrepresented sequences**: es esperable que salgan en amarillo. FastQC compara el %GC observado contra una distribución normal teórica, supuesto que funciona bien para un genoma grande y complejo, pero que **no se cumple en genomas chicos, circulares o de baja diversidad** como este; y con 2000 pares de lecturas de 100 pb sobre un genoma de solo 16.569 pb, la cobertura es tan alta (~24×) que varias lecturas caen exactamente en la misma posición y se ven como "duplicadas/sobrerepresentadas" sin que eso sea un problema real. Van a ver este mismo patrón cada vez que trabajen con genomas virales, plásmidos o, como hoy, mitocondrial.

---
## 4. Alineamiento de secuencias con BWA (40 min)

`bwa` ya está indexado (sección 2.3). `bwa mem` es el algoritmo recomendado para lecturas de 70 pb o más (para lecturas más cortas existen `bwa aln`/`bwa samse`, que no usaremos en este curso).

Un detalle importante: le vamos a pasar un **read group** (`-R`) con `bwa mem`. Es metadata que identifica de qué muestra viene cada lectura (`SM`), qué tecnología se usó (`PL`), etc. Muchas herramientas de llamado de variantes (como GATK) **exigen** que el BAM tenga un read group — conviene acostumbrarse a incluirlo siempre desde el alineamiento.

In [ ]:
%%bash
cd ~/bioinfo/clase4
mkdir -p alineamiento && cd alineamiento

bwa mem -R '@RG\tID:mtDNA_sample\tSM:mtDNA_sample\tPL:ILLUMINA' \
    ../referencia/chrM.fasta \
    ../fastq/mtDNA_sample_R1.fastq.gz ../fastq/mtDNA_sample_R2.fastq.gz \
    > mtDNA_sample.sam

echo "--- Encabezado del SAM ---"
grep "^@" mtDNA_sample.sam

echo
echo "--- Primeros alineamientos (columnas 1-9: nombre, flag, ref, pos, mapQ, CIGAR, ref_mate, pos_mate, tlen) ---"
grep -v "^@" mtDNA_sample.sam | head -5 | cut -f1-9

Como repaso de la clase 3: el `CIGAR` `100M` de un registro significa "100 bases que calzan (match/mismatch) contra la referencia, sin inserciones ni deleciones" — el resultado esperable cuando no hay indeles, como en nuestra simulación.

### 4.1 Decodificando el flag

La columna 2 (*flag*) es un número que codifica varias propiedades del alineamiento a la vez (¿está pareado? ¿mapeó? ¿en qué hebra? ¿es la primera o segunda lectura del par?). En vez de memorizar la tabla de bits, `samtools` trae un decodificador.

In [ ]:
%%bash
cd ~/bioinfo/clase4/alineamiento

echo "--- Flag de la primera lectura del archivo ---"
FLAG=$(grep -v "^@" mtDNA_sample.sam | head -1 | cut -f2)
echo "Flag: $FLAG"
samtools flags "$FLAG"

echo
echo "--- Algunos flags de referencia ---"
samtools flags PAIRED,PROPER_PAIR
samtools flags UNMAP
samtools flags REVERSE,READ1

---
## 5. Procesamiento de archivos SAM/BAM con samtools (45 min)

### 5.1 De SAM a BAM ordenado e indexado

El SAM que generó `bwa mem` no viene ordenado por posición, y es texto plano (pesado). El flujo estándar es: convertir a BAM (binario), **ordenar por posición** e **indexar** — recién ahí el archivo queda listo para explorarlo por región o para pasarlo a un llamador de variantes.

In [ ]:
%%bash
cd ~/bioinfo/clase4/alineamiento

echo "--- SAM -> BAM (samtools view -b) ---"
samtools view -b mtDNA_sample.sam > mtDNA_sample.bam

echo "--- Ordenar por posicion (samtools sort) ---"
samtools sort -o mtDNA_sample.sorted.bam mtDNA_sample.bam

echo "--- Indexar (samtools index) ---"
samtools index mtDNA_sample.sorted.bam

ls -la mtDNA_sample*

echo
echo "--- Comparacion de tamano SAM vs BAM ---"
du -h mtDNA_sample.sam mtDNA_sample.bam mtDNA_sample.sorted.bam

> **En la práctica**, `samtools sort` puede leer un SAM directamente y escribir un BAM ya ordenado en un solo paso: `samtools sort -O bam -o mtDNA_sample.sorted.bam mtDNA_sample.sam`. Hicimos los dos pasos por separado para que quede claro qué hace cada uno; de ahora en adelante pueden usar el atajo.

### 5.2 Estadísticas generales: `flagstat` y `stats`

In [ ]:
%%bash
cd ~/bioinfo/clase4/alineamiento

echo "--- samtools flagstat ---"
samtools flagstat mtDNA_sample.sorted.bam

echo
echo "--- samtools stats (resumen, seccion SN) ---"
samtools stats mtDNA_sample.sorted.bam | grep ^SN | head -12

Con datos simulados sin indeles y sobre una referencia sin regiones repetitivas problemáticas, es normal ver **100% mapeado y 100% properly paired** — con datos reales casi nunca es así (siempre hay algo de contaminación, lecturas quiméricas, regiones difíciles de mapear, etc.).

### 5.3 Filtrar por flag: `-f` (incluir) y `-F` (excluir)

Algunos flags útiles para filtrar: `2` = properly paired, `4` = unmapped, `256` = secondary alignment, `1024` = duplicado (marcado por otra herramienta, ej. `MarkDuplicates`).

In [ ]:
%%bash
cd ~/bioinfo/clase4/alineamiento

echo "--- Total de alineamientos ---"
samtools view -c mtDNA_sample.sorted.bam

echo "--- Solo 'properly paired' (-f 2) ---"
samtools view -c -f 2 mtDNA_sample.sorted.bam

echo "--- Excluyendo no mapeados (-F 4) ---"
samtools view -c -F 4 mtDNA_sample.sorted.bam

### 5.4 Extraer una región (para esto sirve el índice `.bai`)

In [ ]:
%%bash
cd ~/bioinfo/clase4/alineamiento

echo "--- Lecturas que caen entre las posiciones 4800-4900 ---"
samtools view -b mtDNA_sample.sorted.bam NC_012920.1:4800-4900 -o region.bam
samtools index region.bam
samtools view -c region.bam

### 5.5 Mirando la evidencia de una variante con `mpileup`

`samtools mpileup` muestra, base por base, qué observó cada lectura que cubre esa posición — es la forma más directa de "ver a ojo" si hay evidencia de una variante antes de correr un llamador formal (próxima clase).

Vamos a mirar dos posiciones a modo de ejemplo (ustedes explorarán otras en el ejercicio integrador):

In [ ]:
%%bash
cd ~/bioinfo/clase4

echo "--- Posicion NC_012920.1:4827 ---"
samtools mpileup -f referencia/chrM.fasta -r NC_012920.1:4827-4827 alineamiento/mtDNA_sample.sorted.bam 2>/dev/null

echo
echo "--- Posicion NC_012920.1:591 ---"
samtools mpileup -f referencia/chrM.fasta -r NC_012920.1:591-591 alineamiento/mtDNA_sample.sorted.bam 2>/dev/null

La 4ª columna es la profundidad (cuántas lecturas cubren esa base) y la 5ª muestra, por cada lectura, si coincide con la referencia (`.`/`,`) o qué base alternativa observó. En la posición 4827 deberían ver que **prácticamente todas** las lecturas muestran la misma base alternativa (patrón consistente con una variante homocigota/homoplásmica). En la 591 deberían ver una **mezcla** de la base de referencia y una alternativa en proporciones parecidas — el patrón que en mtDNA se interpreta como posible **heteroplasmia** (coexistencia de dos poblaciones de mtDNA en la misma muestra) y en un locus nuclear diploide como heterocigosis.

---
## 6. Ejercicio integrador (en parejas, 15 min)

Van a repetir el flujo completo con **su propio set de lecturas**, de punta a punta, sin mirar todavía el archivo de respuestas de `dwgsim`.

1. Simulen un nuevo set de lecturas con una semilla (`-z`) distinta a la del ejemplo (por ejemplo, usen los últimos dígitos del RUT de alguno de los dos integrantes de la pareja), guardándolo con otro nombre de muestra (ej. `mtDNA_pareja`).
2. Corran el flujo completo: `bwa mem` → `samtools view` → `sort` → `index`.
3. Reporten: `samtools flagstat` (¿qué % quedó *properly paired*?) y cuántas lecturas caen en la región `NC_012920.1:8000-9000` (`samtools view -c`).
4. Elijan **3 posiciones al azar** dentro de `NC_012920.1` y córranles `samtools mpileup`. Para cada una, decidan: ¿parece 100% referencia, una variante homocigota/homoplásmica, o una mezcla (heteroplasmia/heterocigosis)?
5. Recién ahora, abran `mtDNA_pareja.mutations.txt` (el archivo que generó `dwgsim`) y comparen: ¿las posiciones donde `dwgsim` puso una mutación coinciden con lo que ustedes observaron en el `mpileup`? ¿Alguna de sus 3 posiciones al azar cayó justo en una de las mutaciones simuladas, o no?

No hay una única forma "correcta" de organizar los comandos — el objetivo es que ejecuten el flujo de memoria, sin ir copiando celda por celda de las secciones anteriores.

---
## 7. Cierre y resumen

### Conceptos y comandos vistos hoy

```
Herramientas nuevas: fastqc, dwgsim

Referencia:  samtools faidx, bwa index
Simulacion:  dwgsim (-N, -1/-2, -e/-E, -r, -R, -z)
QC:          fastqc
Alineamiento: bwa mem, read groups (-R), flag SAM, samtools flags
SAM/BAM:     samtools view -b, sort, index, flagstat, stats
Filtros:     samtools view -f / -F / -c, extraccion de regiones
Exploracion: samtools mpileup
```

### Lo que dejamos listo para la clase 5

- Un ambiente `alineamiento` con `bwa`, `samtools`, `fastqc` y `dwgsim` instalados y verificados.
- Una referencia mitocondrial indexada (`~/bioinfo/clase4/referencia/chrM.fasta` + índices `.fai` y de `bwa`).
- Un BAM **ordenado e indexado** (`~/bioinfo/clase4/alineamiento/mtDNA_sample.sorted.bam`) — el insumo directo para el llamado de variantes.
- Un archivo de respuesta (`mtDNA_sample.mutations.vcf`) guardado sin revisar en detalle, que vamos a comparar con lo que encuentre un llamador de variantes real.

### Próxima clase: Llamado de variantes I

Vamos a tomar exactamente el BAM que dejamos listo hoy y usar un llamador de variantes (`bcftools mpileup`/`call` o similar) para pasar de "evidencia visual en un `mpileup`" a un archivo **VCF** formal, con genotipos y medidas de confianza — y comparar el resultado contra el archivo de respuestas de `dwgsim` que guardaron hoy.

**Tarea para antes de la próxima clase:**

1. Verifiquen que su BAM ordenado e indexado del ejercicio integrador (`mtDNA_pareja.sorted.bam` + `.bai`) sigue disponible.
2. Si no alcanzaron a terminar el punto 5 del ejercicio integrador (comparar contra `mutations.txt`), termínenlo antes de la próxima clase.
3. Repasen brevemente qué es un flag SAM y qué información da un CIGAR (secciones 4 y clase 3) — la próxima clase los vamos a dar por conocidos.